In [3]:
#  Load processed/preprocessed data
import json
import numpy as np
import pandas as pd
import os

df = pd.read_csv("../transaction_data.csv")
print(df.shape)
df.head()

(6362620, 31)


,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,type_encoded,is_oldbalanceOrg_zero,...,amount_to_oldbalance_dest_ratio,sender_account_emptied,dest_received_large_amount,is_large_transaction,step_bucket,transactions_in_step,is_high_velocity_step,is_dest_high_balance,type_risk_score,suspicious_signal_count
0,1,9839.64,170136.0,160296.36,0.0,0.0,0,0,3,0,...,0.000000,0,0,0,0,2708,0,0,1,1
1,1,1864.28,21249.0,19384.72,0.0,0.0,0,0,3,0,...,0.000000,0,0,0,0,2708,0,0,1,1
2,1,181.00,181.0,0.00,0.0,0.0,1,0,4,0,...,0.000000,1,0,0,0,2708,0,0,3,2
3,1,181.00,181.0,0.00,21182.0,0.0,1,0,1,0,...,0.008545,1,0,0,0,2708,0,0,3,2
4,1,11668.14,41554.0,29885.86,0.0,0.0,0,0,3,0,...,0.000000,0,0,0,0,2708,0,0,1,1


In [4]:
#  Compute config stats (thresholds + step counts map)
config = {}

# 95th percentile threshold for "large transaction"
config["large_threshold_p95"] = float(df["amount"].quantile(0.95))

# median for destination high balance flag
config["median_oldbalanceDest"] = float(df["oldbalanceDest"].median())

# step bin edges (same logic as your feature engineering)
max_step = int(df["step"].max())
bins = np.linspace(0, max_step, 7).tolist()
config["step_bins"] = bins

# step -> transaction count (velocity proxy)
step_counts = df.groupby("step").size().to_dict()
# keys must be str for json
config["step_counts"] = {str(int(k)): int(v) for k, v in step_counts.items()}

# median transactions in a step (for high velocity flag)
config["median_transactions_in_step"] = float(np.median(list(step_counts.values())))

config

{'large_threshold_p95': 518634.19649999996,
 'median_oldbalanceDest': 132705.66499999998,
 'step_bins': [0.0,
  123.83333333333333,
  247.66666666666666,
  371.5,
  495.3333333333333,
  619.1666666666666,
  743.0],
 'step_counts': {'1': 2708,
  '2': 1014,
  '3': 552,
  '4': 565,
  '5': 665,
  '6': 1660,
  '7': 6837,
  '8': 21097,
  '9': 37628,
  '10': 35991,
  '11': 37241,
  '12': 36153,
  '13': 37515,
  '14': 41485,
  '15': 44609,
  '16': 42471,
  '17': 43361,
  '18': 49579,
  '19': 51352,
  '20': 40625,
  '21': 19152,
  '22': 12635,
  '23': 6144,
  '24': 3216,
  '25': 1598,
  '26': 440,
  '27': 41,
  '28': 4,
  '29': 4,
  '30': 8,
  '31': 12,
  '32': 12,
  '33': 23616,
  '34': 30904,
  '35': 29157,
  '36': 39774,
  '37': 34000,
  '38': 31453,
  '39': 23391,
  '40': 34270,
  '41': 36348,
  '42': 41304,
  '43': 45060,
  '44': 38523,
  '45': 18500,
  '46': 12445,
  '47': 8681,
  '48': 5693,
  '49': 764,
  '50': 6,
  '51': 14,
  '52': 8,
  '53': 10,
  '54': 4,
  '55': 12,
  '56': 18,
  '

In [5]:
os.makedirs("../models", exist_ok=True)

out_path = "../models/feature_config.json"
with open(out_path, "w") as f:
    json.dump(config, f, indent=2)

print("Saved:", out_path)

Saved: ../models/feature_config.json


In [6]:
import json, os
import numpy as np
import pandas as pd

pre = pd.read_csv("../data/processed/preprocessed_data.csv")  # has type column ideally
eng = pd.read_csv("../data/processed/feature_engineered_data.csv")  # final training style

print("pre:", pre.shape)
print("eng:", eng.shape)

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/preprocessed_data.csv'

In [ ]:
config = {}

config["large_threshold_p95"] = float(pre["amount"].quantile(0.95))
config["median_oldbalanceDest"] = float(pre["oldbalanceDest"].median())

max_step = int(pre["step"].max())
config["step_bins"] = np.linspace(0, max_step, 7).tolist()

step_counts = pre.groupby("step").size().to_dict()
config["step_counts"] = {str(int(k)): int(v) for k, v in step_counts.items()}
config["median_transactions_in_step"] = float(np.median(list(step_counts.values())))

config


os.makedirs("../models", exist_ok=True)

with open("../models/feature_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Saved: ../models/feature_config.json")

In [ ]:
drop_cols = ["isFraud", "isFlaggedFraud"]  # same as training
X_cols = [c for c in eng.columns if c not in drop_cols]

with open("../models/feature_columns.json", "w") as f:
    json.dump(X_cols, f, indent=2)

print("Saved: ../models/feature_columns.json")
print("Total model features:", len(X_cols))